# 2026-06-10: 뷰티풀수프를 활용한 크롤링

### 1. 뷰티풀수프(BeautifulSoup) 매뉴얼 & 자주 하는 실수
* **`find()`** vs **`find_all()`**: 하나만 찾을 때는 `find`로 가져와 바로 텍스트를 가져올 수 있지만, `find_all`은 데이터를 대괄호 `[...]` 리스트 상자에 담아주므로 **반드시 `for` 반복문**을 돌려야 알맹이를 뺄 수 있음!
* **`.get_text()`의 에러**: 태그에 `.get_text()`를 붙이는 순간 순수한 '글자(문자열)'로 바뀜. 글자 상태에서는 부모(`.parent`)나 형제(`.find_next_sibling()`)를 추적하는 뷰티풀수프 고유 기능을 쓸 수 없으니, **위로 타고 올라간 다음에 마지막에 글자를 가져와야 함!**
* **함수 이름 언더바(`_`)**: `findall`은 없는 문법! 무조건 **`find_all()`**로 언더바를 붙여야 함. class 속성 단독 매칭 시에는 **`class_`**로 언더바 필수.

### 2. datetime 날짜 자동화 매칭 팁
* **`timedelta(days=1)`**: 오늘 날짜에서 하루 전(어제)을 계산할 때는 복수형 옵션인 `days=1`을 기부함.
* **타입 매칭 에러**: `datetime.today()`로 계산한 날짜 객체와 웹사이트에서 긁어온 텍스트(문자열)는 서로 비교할 수 없음. 반드시 날짜 객체 뒤에 **`.strftime('%Y.%m.%d')`**을 붙여 똑같은 글자 형태로 맞춰준 뒤 비교할 것!
* **`in` 문법**: 웹사이트 날짜 뒤에 요일(`(월)`)이 붙거나 보이지 않는 공백이 섞여 있으면 `startswith()` 에러날 수 있으므로, **`if yesterday_str in date_text:`** 처럼 포함 검색(`in`)을 쓰는 것이 실무에서 권장.

### 3. 파이썬 문법과 관련한 주의사항
* **리스트 인덱싱**: 슬라이싱(`num_lst[::2]`)이 이미 홀수를 배달해 주었다면, `print(num_lst[n])`처럼 방 번호로 또 쓰지 말고 변수 `n` 자체를 바로 출력할 것.
* **들여쓰기(Indentation)**: `for`문이나 `if`문 바로 아랫줄은 무조건 `Tab` 키를 눌러 안쪽으로 라인을 밀어주어야 에러가 안 남!

In [3]:
import requests
from bs4 import BeautifulSoup

url = 'http://www.nate.com/'

res = requests.get(url)

print(res.status_code)

200


In [4]:
url = 'https://www.nate.com/'

res = requests.get(url)
html = res.text

# print(res.status_code)

soup = BeautifulSoup(html, 'html.parser')
# print(soup.prettify())

In [10]:
print(soup.title)

<title>네이트</title>


In [12]:
print(soup.title.get_text())

네이트


In [15]:
url = 'https://www.nate.com/'

res = requests.get(url)
html = res.text

# print(res.status_code)

soup = BeautifulSoup(html, 'lxml')  # 속도가 더 빠름
# print(soup.prettify())

print(soup.title)
print(soup.title.get_text())

print(soup.a)  # soup 라는 객체에서 처음 발견되는 a 태그 가져옴
print('a태그의 속성값:', soup.a.attrs)  # a 태그의 속성 알려줘
print('a태그 중에 href 속성값:', soup.a['href']) 


print('-'*30)
print(soup.find('a', attrs={'class' : 'link'}))

<title>네이트</title>
네이트
<a href="#search"><strong>통합검색 바로가기</strong></a>
a태그의 속성값: {'href': '#search'}
a태그 중에 href 속성값: #search
------------------------------
<a class="link" href="https://pann.nate.com/talk/375454083" onmousedown="nc('PAO80');objPannArea.clickContents(0);">튜닝한 기아 소형차(?) 타는 썸남... 정떨어지는데 제가 예민한가요?</a>


In [17]:
print(soup.find('div', attrs={'class':'slide-content'}))

<div class="slide-content">
<span class="num_rank">6</span>
<a class="ik" href="https://news.nate.com/search?q=월드컵 일정" onclick="newsBox.clickSearchKeyword('월드컵 일정');return false;" onmousedown="nc('MEH21')">
<span class="txt_rank">월드컵 일정</span>
<span class="fc new"><span class="nHide">new</span></span>
</a>
</div>


In [18]:
# 전체 HTML 중에서 태그 이름은 'a'이고, 속성(attrs) 중 class 이름이 'link'인 첫 번째 태그 요소를 정확히 찾아 출력함
print(soup.find('a', attrs={'class' : 'link'}))

<a class="link" href="https://pann.nate.com/talk/375454083" onmousedown="nc('PAO80');objPannArea.clickContents(0);">튜닝한 기아 소형차(?) 타는 썸남... 정떨어지는데 제가 예민한가요?</a>


In [20]:
rank1_div = print(soup.find('div', attrs={'class':'slide-content'}))
print(rank1_div)

<div class="slide-content">
<span class="num_rank">6</span>
<a class="ik" href="https://news.nate.com/search?q=월드컵 일정" onclick="newsBox.clickSearchKeyword('월드컵 일정');return false;" onmousedown="nc('MEH21')">
<span class="txt_rank">월드컵 일정</span>
<span class="fc new"><span class="nHide">new</span></span>
</a>
</div>
None


In [22]:
rank1_div = soup.find('div', attrs={'class':'slide-content'})
rank1 = rank1_div.parent

print(rank1)

<li>
<div class="slide-content">
<span class="num_rank">6</span>
<a class="ik" href="https://news.nate.com/search?q=월드컵 일정" onclick="newsBox.clickSearchKeyword('월드컵 일정');return false;" onmousedown="nc('MEH21')">
<span class="txt_rank">월드컵 일정</span>
<span class="fc new"><span class="nHide">new</span></span>
</a>
</div>
</li>


In [25]:
rank2 = rank1.next_sibling
print(rank2) # 공백이 나옴

In [26]:
rank2 = rank1.next_sibling.next_sibling   # 공백이 있어서 한번 더 써줌
print(rank2)

<li>
<div class="slide-content">
<span class="num_rank">7</span>
<a class="ik" href="https://news.nate.com/search?q=곽재선 KG그룹 회장" onclick="newsBox.clickSearchKeyword('곽재선 KG그룹 회장');return false;" onmousedown="nc('MEH22')">
<span class="txt_rank">곽재선 KG그룹 회장</span>
<span class="fc same"><span class="nHide">동일</span></span>
</a>
</div>
</li>


In [28]:
rank3 = rank2.next_sibling.next_sibling
print(rank3)

<li>
<div class="slide-content">
<span class="num_rank">8</span>
<a class="ik" href="https://news.nate.com/search?q=종합특검 이전 기소" onclick="newsBox.clickSearchKeyword('종합특검 이전 기소');return false;" onmousedown="nc('MEH23')">
<span class="txt_rank">종합특검 이전 기소</span>
<span class="fc new"><span class="nHide">new</span></span>
</a>
</div>
</li>


In [30]:
rank2 = rank3.previous_sibling.previous_sibling
print(rank2)

<li>
<div class="slide-content">
<span class="num_rank">7</span>
<a class="ik" href="https://news.nate.com/search?q=곽재선 KG그룹 회장" onclick="newsBox.clickSearchKeyword('곽재선 KG그룹 회장');return false;" onmousedown="nc('MEH22')">
<span class="txt_rank">곽재선 KG그룹 회장</span>
<span class="fc same"><span class="nHide">동일</span></span>
</a>
</div>
</li>


In [36]:
rank1 = rank2.previous_sibling.previous_sibling
print(rank1)

<li>
<div class="slide-content">
<span class="num_rank">6</span>
<a class="ik" href="https://news.nate.com/search?q=월드컵 일정" onclick="newsBox.clickSearchKeyword('월드컵 일정');return false;" onmousedown="nc('MEH21')">
<span class="txt_rank">월드컵 일정</span>
<span class="fc new"><span class="nHide">new</span></span>
</a>
</div>
</li>


In [34]:
ol = rank2.parent
print(ol)

<ol class="isKeywordList" id="olLiveIssueKeyword">
<li>
<div class="slide-content">
<span class="num_rank">6</span>
<a class="ik" href="https://news.nate.com/search?q=월드컵 일정" onclick="newsBox.clickSearchKeyword('월드컵 일정');return false;" onmousedown="nc('MEH21')">
<span class="txt_rank">월드컵 일정</span>
<span class="fc new"><span class="nHide">new</span></span>
</a>
</div>
</li>
<li>
<div class="slide-content">
<span class="num_rank">7</span>
<a class="ik" href="https://news.nate.com/search?q=곽재선 KG그룹 회장" onclick="newsBox.clickSearchKeyword('곽재선 KG그룹 회장');return false;" onmousedown="nc('MEH22')">
<span class="txt_rank">곽재선 KG그룹 회장</span>
<span class="fc same"><span class="nHide">동일</span></span>
</a>
</div>
</li>
<li>
<div class="slide-content">
<span class="num_rank">8</span>
<a class="ik" href="https://news.nate.com/search?q=종합특검 이전 기소" onclick="newsBox.clickSearchKeyword('종합특검 이전 기소');return false;" onmousedown="nc('MEH23')">
<span class="txt_rank">종합특검 이전 기소</span>
<span class="fc new">

In [37]:
ol = rank1.parent
print(ol)

<ol class="isKeywordList" id="olLiveIssueKeyword">
<li>
<div class="slide-content">
<span class="num_rank">6</span>
<a class="ik" href="https://news.nate.com/search?q=월드컵 일정" onclick="newsBox.clickSearchKeyword('월드컵 일정');return false;" onmousedown="nc('MEH21')">
<span class="txt_rank">월드컵 일정</span>
<span class="fc new"><span class="nHide">new</span></span>
</a>
</div>
</li>
<li>
<div class="slide-content">
<span class="num_rank">7</span>
<a class="ik" href="https://news.nate.com/search?q=곽재선 KG그룹 회장" onclick="newsBox.clickSearchKeyword('곽재선 KG그룹 회장');return false;" onmousedown="nc('MEH22')">
<span class="txt_rank">곽재선 KG그룹 회장</span>
<span class="fc same"><span class="nHide">동일</span></span>
</a>
</div>
</li>
<li>
<div class="slide-content">
<span class="num_rank">8</span>
<a class="ik" href="https://news.nate.com/search?q=종합특검 이전 기소" onclick="newsBox.clickSearchKeyword('종합특검 이전 기소');return false;" onmousedown="nc('MEH23')">
<span class="txt_rank">종합특검 이전 기소</span>
<span class="fc new">

In [42]:
print(rank1)
ol = rank1.parent

real_new = soup.find('span', string='월드컵')
print(real_new)

<li>
<div class="slide-content">
<span class="num_rank">6</span>
<a class="ik" href="https://news.nate.com/search?q=월드컵 일정" onclick="newsBox.clickSearchKeyword('월드컵 일정');return false;" onmousedown="nc('MEH21')">
<span class="txt_rank">월드컵 일정</span>
<span class="fc new"><span class="nHide">new</span></span>
</a>
</div>
</li>
None


In [43]:
import re

# 정규표현식 패턴을 string 옵션에 집어넣어 포함 검색으로 변경
real_new = soup.find('span', string=re.compile('월드컵'))

print(real_new)

<span class="txt_rank">월드컵 일정</span>


In [44]:
import requests
from bs4 import BeautifulSoup

url = "https://www.melon.com/chart/index.htm"
headers = {'User-Agent' : 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36'}


res = requests.get(url, headers=headers)
html = res.text
# print(res.status_code)
soup = BeautifulSoup(html, "lxml") # lxml 속도 좀 더 빠르다 - html-parser
print(soup.title)
print(soup.title.get_text())

<title>멜론차트&gt;TOP100&gt;멜론</title>
멜론차트>TOP100>멜론


In [48]:
# 전체 HTML 중에서 태그 이름은 'div'이고, class 속성명이 'wrap_song_info'인 모든 곡 정보를 '리스트(List)' 형태로 musics 변수에 저장
musics = soup.find_all('div', attrs={'class':'wrap_song_info'})

print(musics[0])   # 필요한 정보
print('-'*30)
print(musics[1])   # 불필요한 정보

<div class="wrap_song_info">
<div class="ellipsis rank01"><span>
<a href="javascript:melon.play.playSong('1000002721',602024048);" title="갑자기 재생">갑자기</a>
</span></div><br/>
<div class="ellipsis rank02">
<a href="/artist/detail.htm?artistId=960251" title="아이오아이 (I.O.I) - 페이지 이동">아이오아이 (I.O.I)</a><span class="checkEllipsis" style="display:none"><a href="/artist/detail.htm?artistId=960251" title="아이오아이 (I.O.I) - 페이지 이동">아이오아이 (I.O.I)</a></span>
</div>
</div>
------------------------------
<div class="wrap_song_info">
<div class="ellipsis rank03">
<a href="/album/detail.htm?albumId=13399673" title="I.O.I 3rd MINI ALBUM [I.O.I : LOOP] - 페이지 이동">I.O.I 3rd MINI ALBUM [I.O.I : LOOP]</a>
</div>
</div>


In [49]:
print(musics[2])   # 필요한 정보
print('-'*30)
print(musics[3])   # 불필요한 정보

<div class="wrap_song_info">
<div class="ellipsis rank01"><span>
<a href="javascript:melon.play.playSong('1000002721',601807965);" title="REDRED 재생">REDRED</a>
</span></div><br/>
<div class="ellipsis rank02">
<a href="/artist/detail.htm?artistId=4491260" title="CORTIS (코르티스) - 페이지 이동">CORTIS (코르티스)</a><span class="checkEllipsis" style="display:none"><a href="/artist/detail.htm?artistId=4491260" title="CORTIS (코르티스) - 페이지 이동">CORTIS (코르티스)</a></span>
</div>
</div>
------------------------------
<div class="wrap_song_info">
<div class="ellipsis rank03">
<a href="/album/detail.htm?albumId=13338387" title="GREENGREEN - 페이지 이동">GREENGREEN</a>
</div>
</div>


In [50]:
# 반복문 활용!

for music in musics :
    print(music.a.get_text())

갑자기
I.O.I 3rd MINI ALBUM [I.O.I : LOOP]
REDRED
GREENGREEN
It′s Me
MAMIHLAPINATAPAI
LEMONADE
LEMONADE - The 2nd Album
소문의 낙원
개화
캐치 캐치
LOVE CATCHER
기쁨, 슬픔, 아름다운 마음
개화
RUDE!
RUDE!
사랑하게 될 거야
이상비행
LOVE ATTACK
SCENEDROME
Heavy Serenade
Heavy Serenade
0+0
자몽살구클럽
Drowning
OO-LI
WDA (Whole Different Animal) (Feat. G-DRAGON)
LEMONADE - The 2nd Album
404 (New Era)
Delulu Pack
BANG BANG
REVIVE+
Good Goodbye
Good Goodbye
SWIM
ARIRANG
타임캡슐
타임캡슐
Popcorn
성장 - THE 3RD MINI ALBUM
Blue Valentine
Blue Valentine
너에게 닿기를
너에게 닿기를
멸종위기사랑
EROS
어떻게 이별까지 사랑하겠어, 널 사랑하는 거지
항해
어제보다 슬픈 오늘
어제보다 슬픈 오늘
VIRAL
HOME
Body to Body
ARIRANG
뛰어(JUMP)
뛰어(JUMP)
너의 모든 순간
별에서 온 그대 OST Part.7
그대 작은 나의 세상이 되어
부재
Golden
KPop Demon Hunters (Soundtrack from the Netflix Film)
모르시나요(PROD.로코베리)
모르시나요
소나기
선재 업고 튀어 OST Part 1
Whiplash
Whiplash - The 5th Mini Album
toxic till the end
rosie
내게 사랑이 뭐냐고 물어본다면
내게 사랑이 뭐냐고 물어본다면
BOOMPALA
'PUREFLOW' pt.1
천상연
천상연 (웹툰 '선녀외전' X 이창섭 (LEE CHANGSUB))
HOME SWEET HOME (feat. 태양, 대성)
HOME SWEET HOME (feat. 

In [51]:
# 반복문 활용! - 필요한 정보만!

for music in musics[::2] :
    print(music.a.get_text())

갑자기
REDRED
It′s Me
LEMONADE
소문의 낙원
캐치 캐치
기쁨, 슬픔, 아름다운 마음
RUDE!
사랑하게 될 거야
LOVE ATTACK
Heavy Serenade
0+0
Drowning
WDA (Whole Different Animal) (Feat. G-DRAGON)
404 (New Era)
BANG BANG
Good Goodbye
SWIM
타임캡슐
Popcorn
Blue Valentine
너에게 닿기를
멸종위기사랑
어떻게 이별까지 사랑하겠어, 널 사랑하는 거지
어제보다 슬픈 오늘
VIRAL
Body to Body
뛰어(JUMP)
너의 모든 순간
그대 작은 나의 세상이 되어
Golden
모르시나요(PROD.로코베리)
소나기
Whiplash
toxic till the end
내게 사랑이 뭐냐고 물어본다면
BOOMPALA
천상연
HOME SWEET HOME (feat. 태양, 대성)
한 페이지가 될 수 있게
Seven (feat. Latto) - Clean Ver.
청춘만화
like JENNIE
그대만 있다면 (여름날 우리 X 너드커넥션 (Nerd Connection))
HAPPY
봄날
2.0
모든 날, 모든 순간 (Every day, Every Moment)
주저하는 연인들을 위해
BLACKHOLE
STYLE
SPAGHETTI (feat. j-hope of BTS)
사랑인가 봐
나는 반딧불
사랑은 늘 도망가
예뻤어
Hooligan
APT.
FAMOUS
첫 만남은 계획대로 되지 않아
청혼하지 않을 이유를 못 찾았어
입춘
오늘만 I LOVE YOU
Dynamite
Welcome to the Show
REBEL HEART
사랑은 봄비처럼...이별은 겨울비처럼...
달리 표현할 수 없어요
Love Love Love (Feat. Yoong Jin Of Casker)
눈을 감아도(2026)
ONE MORE TIME
FYA
다정히 내 이름을 부르면
시작의 아이 ❍
Flower
OVERDRIVE
TICK TOCK (Feat. ZICO) (Prod. by ZIC

In [52]:
# 짝수 반복 / 홀수 반복
# 0부터 시작함!

num_lst = [1,2,3,4,5,6,7,8,9,10]
print(num_lst)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [53]:
for n in num_lst[::2]:
    print(num_lst[n])

2
4
6
8
10


In [54]:
num_lst = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# 처음부터 끝까지 2칸 간격으로 데이터(1, 3, 5, 7, 9)를 뽑아 n에 하나씩 담아라
for n in num_lst[::2]:
    print(n)  # 🎯 여기를 수정! n이 곧 내가 찾던 진짜 홀수 데이터임

1
3
5
7
9


In [70]:
# 기본형!

import requests
from bs4 import BeautifulSoup

url = "https://www.joongang.co.kr/sports"
# user-agent
headers = {"User-Agent" : "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36" }

res = requests.get(url, headers=headers)
html = res.text
print("응답코드: ", res.status_code)

# 모듈 활용
soup = BeautifulSoup(html, "lxml") # lxml 속도 좀 더 빠르다 - html-parser

sports_ul = soup.find("ul", attrs = {'class':'story_list'})  # 정보 저장
sports_recents = sports_ul.find_all('h2', attrs={'class':'headline'})

print(sports_recents[0].a.get_text())
print(sports_recents[0].a['href'])

# 반복문 활용

for recent in sports_recents :
    print(recent.a.get_text())
    print(recent.a['href'])

응답코드:  200

한국여자오픈 출격하는 이동은 “김효주 선배처럼 우승할래요”

https://www.joongang.co.kr/article/25435245

한국여자오픈 출격하는 이동은 “김효주 선배처럼 우승할래요”

https://www.joongang.co.kr/article/25435245

체코전 베스트11 홍명보의 고민은?… 이황재 해설위원 분석

https://www.joongang.co.kr/article/25435238

광주의 김도영 vs 잠실의 오스틴, 홈런왕 레이스 점입가경

https://www.joongang.co.kr/article/25435214

체코, 한국전 앞두고 훈련 전면 비공개...필승 전략 준비하나

https://www.joongang.co.kr/article/25435200

“영화 ‘쿨러닝’ 같다”... 낡은 버스 타고 월드컵 치르는 ‘인구 15만’ 퀴라소 국대

https://www.joongang.co.kr/article/25435165

[북중미 월드컵 H조] ‘무적함대’ 스페인과 남미·중동 맹주들의 혈전

https://www.joongang.co.kr/article/25435145

“흐름 안 끊는 스타일”…홍명보호 체코전 주심은 ‘이집트 변호사’

https://www.joongang.co.kr/article/25435141

[북중미 월드컵 G조] 벨기에 ‘황금세대’와 대륙별 맹주의 예측불허 승부

https://www.joongang.co.kr/article/25435116

슬슬 감추기 시작한다…체코·남아공 보안 강화

https://www.joongang.co.kr/article/25434977

잠실구장 마지막 올스타전, ‘주인’ 두산·LG가 주인공

https://www.joongang.co.kr/article/25434976

광화문광장서 다시 “대∼한민국”…월드컵 한국경기 거리응원

https://www.joongang.co.kr/article/25434877

월드컵 앞두고 아빠 된 골

In [66]:
print(sports_recents[0])

<h2 class="headline">
<a data-evnt-act="move:중앙섹션-A-최신기사-기본 스토리 리스트" data-evnt-ctg="area:중앙|스포츠" data-evnt-lbl="1" href="https://www.joongang.co.kr/article/25435245">
한국여자오픈 출격하는 이동은 “김효주 선배처럼 우승할래요”
</a>
</h2>


In [73]:
# 깔끔하게 정리
for recent in sports_recents :
    title = recent.a.get_text()
    link = recent.a['href']
    print(title, link)


한국여자오픈 출격하는 이동은 “김효주 선배처럼 우승할래요”
 https://www.joongang.co.kr/article/25435245

체코전 베스트11 홍명보의 고민은?… 이황재 해설위원 분석
 https://www.joongang.co.kr/article/25435238

광주의 김도영 vs 잠실의 오스틴, 홈런왕 레이스 점입가경
 https://www.joongang.co.kr/article/25435214

체코, 한국전 앞두고 훈련 전면 비공개...필승 전략 준비하나
 https://www.joongang.co.kr/article/25435200

“영화 ‘쿨러닝’ 같다”... 낡은 버스 타고 월드컵 치르는 ‘인구 15만’ 퀴라소 국대
 https://www.joongang.co.kr/article/25435165

[북중미 월드컵 H조] ‘무적함대’ 스페인과 남미·중동 맹주들의 혈전
 https://www.joongang.co.kr/article/25435145

“흐름 안 끊는 스타일”…홍명보호 체코전 주심은 ‘이집트 변호사’
 https://www.joongang.co.kr/article/25435141

[북중미 월드컵 G조] 벨기에 ‘황금세대’와 대륙별 맹주의 예측불허 승부
 https://www.joongang.co.kr/article/25435116

슬슬 감추기 시작한다…체코·남아공 보안 강화
 https://www.joongang.co.kr/article/25434977

잠실구장 마지막 올스타전, ‘주인’ 두산·LG가 주인공
 https://www.joongang.co.kr/article/25434976

광화문광장서 다시 “대∼한민국”…월드컵 한국경기 거리응원
 https://www.joongang.co.kr/article/25434877

월드컵 앞두고 아빠 된 골키퍼 김승규 “딸에게 좋은 선물 줘야죠”
 https://www.joongang.co.kr/article/25434867

SOOP, 감스트와 ‘2026 FIFA

In [82]:
# 날짜 정보 찾기

news_dates = sports_ul.find_all('p', attrs={'class':'date'})

print(news_dates[0])

<p class="date">2026.06.09 16:15</p>


In [81]:
# 오늘 날짜 가져오기

from datetime import datetime

today = datetime.today().strftime('%Y.%m.%d')
print(today)


# 날짜 정보 찾기

news_dates = sports_ul.find_all('p', attrs={'class':'date'})

# print(news_dates[0])

for news_date in news_dates:
    print(news_date.get_text(strip=True))  #strip=True 공백제거

2026.06.09
<p class="date">2026.06.09 16:15</p>
2026.06.09 16:15
2026.06.09 16:05
2026.06.09 15:30
2026.06.09 15:05
2026.06.09 13:57
2026.06.09 12:44
2026.06.09 12:04
2026.06.09 11:10
2026.06.09 00:01
2026.06.09 00:01
2026.06.08 16:50
2026.06.08 16:36
2026.06.08 16:07
2026.06.08 15:00
2026.06.08 14:43
2026.06.08 14:35
2026.06.08 14:23
2026.06.08 09:15
2026.06.08 00:12
2026.06.08 00:02
2026.06.08 00:02
2026.06.08 00:02
2026.06.08 00:01
2026.06.07 21:06


In [87]:
# 오늘 날짜 가져오기

from datetime import datetime

today = datetime.today().strftime('%Y.%m.%d')
print(today)

# 오늘 날짜에 발행된 기사만 가져오기
for news_date in news_dates :
    temp_date = news_date.get_text(strip=True)
    
    if(temp_date.startswith(today)):   # 앞의 문자열 안에 동일하게 시작하는 문자열이 있는가?
        print(news_date.get_text(strip=True))

# 기사 크롤링의 경우: 제목, 날짜, 링크

2026.06.09
2026.06.09 16:15
2026.06.09 16:05
2026.06.09 15:30
2026.06.09 15:05
2026.06.09 13:57
2026.06.09 12:44
2026.06.09 12:04
2026.06.09 11:10
2026.06.09 00:01
2026.06.09 00:01


In [92]:
# 어제 날짜

from datetime import datetime, timedelta

yes_day = datetime.today() - timedelta(days=1)

print(yes_day)
print(yes_day.strftime('%Y.%m.%d'))

2026-06-08 16:51:34.730768
2026.06.08


In [132]:
# 데이터랩

url = 'https://datalab.naver.com/'

res = requests.get(url)
html = res.text

print(res.status_code)

soup = BeautifulSoup(html, 'lxml')  # 속도가 더 빠름

print(soup.find('span', attrs={'class' : 'title_cell'}))


200
<span class="title_cell">2026.05.28.(목)</span>


In [133]:

data_dates = soup.find_all('span', attrs={'class' : 'title_cell'})

# print(news_dates[0])

for date in data_dates:
    print(date.get_text(strip=True)) #strip=True 공백제거

2026.05.28.(목)
2026.05.29.(금)
2026.05.30.(토)
2026.05.31.(일)
2026.06.01.(월)
2026.06.02.(화)
2026.06.03.(수)
2026.06.04.(목)
2026.06.05.(금)
2026.06.06.(토)
2026.06.07.(일)
2026.06.08.(월)




In [141]:
yes_date = data_dates[11]
print(yes_date)

<span class="title_cell">2026.06.08.(월)</span>


In [143]:
rank = yes_date.parent.parent
print(rank)

<div class="rank_inner v2">
<strong class="rank_title">
<span class="title_cell">2026.06.08.(월)</span>
</strong>
<div class="rank_scroll" style="left:0px; width:5000px;">
<ul class="rank_list">
<li class="list">
<a class="list_area" href="/shoppingInsight/sKeyword.naver?keyword=%EC%9B%90%ED%94%BC%EC%8A%A4">
<em class="num">1</em>
<span class="title">원피스</span>
</a>
</li>
<li class="list">
<a class="list_area" href="/shoppingInsight/sKeyword.naver?keyword=%EB%B8%94%EB%9D%BC%EC%9A%B0%EC%8A%A4">
<em class="num">2</em>
<span class="title">블라우스</span>
</a>
</li>
<li class="list">
<a class="list_area" href="/shoppingInsight/sKeyword.naver?keyword=%EC%97%AC%EB%A6%84%EC%9B%90%ED%94%BC%EC%8A%A4">
<em class="num">3</em>
<span class="title">여름원피스</span>
</a>
</li>
<li class="list">
<a class="list_area" href="/shoppingInsight/sKeyword.naver?keyword=%EC%97%AC%EB%A6%84%EA%B0%80%EB%94%94%EA%B1%B4">
<em class="num">4</em>
<span class="title">여름가디건</span>
</a>
</li>
<li class="list">
<a class="list_are

In [153]:
category = rank.find_all('span', attrs = {'class' : 'title'})

for cg in category:
    print(cg.get_text(strip=True))

원피스
블라우스
여름원피스
여름가디건
티셔츠
여름블라우스
에고이스트
지고트원피스
남자반팔티
시슬리원피스


In [126]:
from datetime import datetime, timedelta

yes_day = datetime.today() - timedelta(days=1)

print(yes_day)
print(yes_day.strftime('%Y.%m.%d'))

2026-06-08 17:15:43.060190
2026.06.08


In [155]:
# 코드 정리

url = 'https://datalab.naver.com/'

res = requests.get(url)
html = res.text

print(res.status_code)

soup = BeautifulSoup(html, 'lxml')  # 속도가 더 빠름

# print(soup.find('span', attrs={'class' : 'title_cell'}))

data_dates = soup.find_all('span', attrs={'class' : 'title_cell'})


# for date in data_dates:
#    print(date.get_text(strip=True)) #strip=True 공백제거

yes_date = data_dates[11]
# print(yes_date)

rank = yes_date.parent.parent
# print(rank)

category = rank.find_all('span', attrs = {'class' : 'title'})

for cg in category:
    print(cg.get_text(strip=True))

200
원피스
블라우스
여름원피스
여름가디건
티셔츠
여름블라우스
에고이스트
지고트원피스
남자반팔티
시슬리원피스


In [168]:
# 예시

from datetime import datetime, timedelta

yesterday = (datetime.today() - timedelta(days=1)).strftime('%Y.%m.%d')

date_spans = soup.find_all('span', class_= 'title_cell')

for date_span in date_spans:
    date_text = date_span.get_text(strip=True)
    if date_text.startswith(yes_day):

        # strong 태그
        strong_tag = date_span.find_parent('strong')

        # 다음 형제 div
        rank_div = strong_tag.next_sibling.next_sibling
        #rank_div = strong_tag.find_next_sibling('div')

        #li 목록
        li_list = rank_div.find_all('li')
        print(f"날짜: {date_text}")

        for li in li_list:
            rank = li.find('em', class_='num').get_text(strip=True)
            keyword = li.find('span', class_ ='title').get_text(strip=True)

            print(rank, keyword)

        break




날짜: 2026.06.08.(월)
1 원피스
2 블라우스
3 여름원피스
4 여름가디건
5 티셔츠
6 여름블라우스
7 에고이스트
8 지고트원피스
9 남자반팔티
10 시슬리원피스


In [172]:
# 예시

from datetime import datetime, timedelta

yesterday = (datetime.today() - timedelta(days=1)).strftime('%Y.%m.%d')

date_spans = soup.find_all('span', class_= 'title_cell')

for date_span in date_spans:
    date_text = date_span.get_text(strip=True)
    if date_text.startswith(yesterday):
        print(yesterday)
        rank = yes_date.parent.parent

        category = rank.find_all('span', attrs = {'class' : 'title'})

        for cg in category:
            print(cg.get_text(strip=True))

2026.06.08
원피스
블라우스
여름원피스
여름가디건
티셔츠
여름블라우스
에고이스트
지고트원피스
남자반팔티
시슬리원피스
